# Rotten Fruit Detector - SageMaker Studio Training

This notebook trains SSD EfficientDet D5 using the shared code in `src/`. Expected zip contents:

```text
dataset/
  train/*.jpg
  train/annotations.csv
  valid/*.jpg
  valid/annotations.csv
  test/*.jpg
  test/annotations.csv
```

In [ ]:
import sys
from pathlib import Path

import boto3
import sagemaker
import tensorflow as tf

session = sagemaker.Session()
bucket = session.default_bucket()

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Region:", session.boto_region_name)
print("Default bucket:", bucket)

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

from notebook_utils import download_s3_file, extract_dataset_zip, find_dataset_root
from train import prepare_training_data, train_evaluate_export

In [ ]:
S3_DATA_ZIP_URI = None
LOCAL_DATA_ZIP_PATH = PROJECT_DIR / "dataset.zip"
EXTRACT_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "artifacts"
S3_OUTPUT_PREFIX = "rotten-fruit-detection/artifacts"
TF_MODELS_DIR = PROJECT_DIR / "tensorflow_models"
PRETRAINED_MODELS_DIR = PROJECT_DIR / "pretrained_models"
MODEL_DIR = PROJECT_DIR / "training" / "ssd_efficientdet_d5"
EXPORT_DIR = OUTPUT_DIR / "ssd_efficientdet_d5_saved_model"

NUM_TRAIN_STEPS = 5000
NUM_EVAL_STEPS = 500
BATCH_SIZE = 2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if S3_DATA_ZIP_URI:
    download_s3_file(S3_DATA_ZIP_URI, LOCAL_DATA_ZIP_PATH)

if LOCAL_DATA_ZIP_PATH.exists():
    LOCAL_DATA_DIR = extract_dataset_zip(LOCAL_DATA_ZIP_PATH, EXTRACT_DIR)
else:
    LOCAL_DATA_DIR = find_dataset_root(PROJECT_DIR / "data")

print("Dataset root:", LOCAL_DATA_DIR)
print("Artifacts:", OUTPUT_DIR)

In [ ]:
dataset, summary = prepare_training_data(
    data_dir=LOCAL_DATA_DIR,
    output_dir=OUTPUT_DIR,
    export_tensorflow_records=True,
)

summary

In [ ]:
result = train_evaluate_export(
    data_dir=LOCAL_DATA_DIR,
    output_dir=OUTPUT_DIR,
    tf_models_dir=TF_MODELS_DIR,
    pretrained_models_dir=PRETRAINED_MODELS_DIR,
    model_dir=MODEL_DIR,
    export_dir=EXPORT_DIR,
    num_train_steps=NUM_TRAIN_STEPS,
    num_eval_steps=NUM_EVAL_STEPS,
    batch_size=BATCH_SIZE,
)

result

In [ ]:
artifacts_s3_uri = session.upload_data(
    path=str(OUTPUT_DIR),
    bucket=bucket,
    key_prefix=S3_OUTPUT_PREFIX,
)

print("Artifacts S3 URI:", artifacts_s3_uri)